# Analyse de sentiments sur des tweets

Dans ce TP, nous allons classer des tweets en trois catégories : **négatif**, **neutre** ou **positif**.

Nous utiliserons le jeu de données [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) (tâche *sentiment*) : environ 60 000 tweets en anglais, annotés à la main.

Au programme :

1. tokeniser les tweets avec spaCy et construire un vocabulaire ;
2. préparer les batchs avec un `Dataset` et un `DataLoader` PyTorch (padding compris) ;
3. entraîner un LSTM bidirectionnel avec une boucle d'entraînement PyTorch écrite à la main ;
4. comparer le résultat à une référence TF-IDF et à un transformeur pré-entraîné.

*Pensez à activer le GPU : Exécution > Modifier le type d'exécution > GPU.*

## Imports

In [ ]:
!pip install -q datasets

In [ ]:
import collections
import copy

import matplotlib.pyplot as plt
import numpy
import pandas
import spacy
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, recall_score
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs sur : {device}")
torch.manual_seed(0)

## Chargement des données

La bibliothèque `datasets` de Hugging Face télécharge le jeu de données déjà découpé en entraînement, validation et test.

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_names = ["négatif", "neutre", "positif"]
print(tweets)

*Explorez les données :*

- *Affichez une dizaine de tweets de l'ensemble d'entraînement avec leur label. Comment ont été traités les noms d'utilisateurs ?*
- *Les classes sont-elles équilibrées ? Quelle conséquence pour le choix de la métrique ?*
- *Quelle est la longueur typique d'un tweet, en mots ?*

*Astuce : `tweets["train"].to_pandas()` donne une `DataFrame` pandas.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
train_df = tweets["train"].to_pandas()
train_df["sentiment"] = train_df.label.map(dict(enumerate(label_names)))
print(train_df.sample(10, random_state=0)[["sentiment", "text"]].to_string())

train_df.sentiment.value_counts().plot.bar(title="Répartition des classes")
plt.show()

train_df.text.str.split().str.len().plot.hist(bins=30, title="Longueur des tweets (mots)")
plt.show()

- Les noms d'utilisateurs sont remplacés par `@user` : les tweets sont anonymisés.
- La classe *neutre* est la plus fréquente, la classe *négatif* la plus rare : la justesse (*accuracy*) favoriserait un modèle qui prédit surtout *neutre*. La métrique officielle de TweetEval est le **rappel moyen par classe** (*macro recall*), qui donne le même poids à chaque classe.
- Un tweet fait une vingtaine de mots, rarement plus de 40.

## Référence : TF-IDF et régression logistique

Avant d'entraîner un réseau de neurones, il est toujours utile de mesurer ce qu'obtient un modèle simple. Si le réseau ne fait pas mieux, il est inutile !

*Entraînez une régression logistique sur une représentation TF-IDF des tweets (`TfidfVectorizer`, `LogisticRegression(max_iter=1000)`) et calculez son rappel moyen par classe sur l'ensemble de validation avec `recall_score(..., average="macro")`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
vectorizer = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(tweets["train"]["text"])
X_val_tfidf = vectorizer.transform(tweets["validation"]["text"])

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_tfidf, tweets["train"]["label"])
baseline_recall = recall_score(tweets["validation"]["label"],
                               baseline.predict(X_val_tfidf), average="macro")
print(f"Rappel moyen TF-IDF : {baseline_recall:.3f}")

## Tokenisation avec spaCy

Pour un réseau récurrent, chaque tweet doit devenir une séquence d'indices de mots. Nous utilisons le tokeniseur de spaCy, qui gère la ponctuation, les émoticônes et les contractions bien mieux qu'un simple `split()`.

`spacy.blank("en")` crée un pipeline anglais vide : il ne contient que le tokeniseur, ce qui est très rapide.

In [ ]:
nlp = spacy.blank("en")


def tokenize(text: str) -> list[str]:
  return [token.lower_ for token in nlp.tokenizer(text) if not token.is_space]


print(tokenize(tweets["train"][0]["text"]))

## Construction du vocabulaire

*Construisez le vocabulaire à partir de l'ensemble d'entraînement **seulement** :*

- *l'indice `0` est réservé au padding (`<pad>`), l'indice `1` aux mots inconnus (`<unk>`) ;*
- *on ne garde que les mots vus au moins 2 fois ;*
- *codez la fonction `numericalize(text) -> list[int]` qui transforme un tweet en liste d'indices.*

*Pourquoi ne pas utiliser les ensembles de validation et de test pour construire le vocabulaire ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
counts = collections.Counter(
    word for text in tweets["train"]["text"] for word in tokenize(text))
vocab = {"<pad>": 0, "<unk>": 1}
for word, count in counts.most_common():
  if count >= 2:
    vocab[word] = len(vocab)
print(f"Taille du vocabulaire : {len(vocab)}")


def numericalize(text: str) -> list[int]:
  return [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)]


print(numericalize(tweets["train"][0]["text"]))

Le vocabulaire fait partie du modèle : le construire avec les données de validation ou de test ferait « fuiter » de l'information sur ces données, et l'évaluation serait trop optimiste.

## `Dataset` et `DataLoader`

*Codez :*

- *une classe `TweetDataset(Dataset)` dont `__getitem__` renvoie le couple `(tenseur d'indices, label)` ;*
- *une fonction `collate(batch)` qui complète les séquences avec du padding (`pad_sequence`) et renvoie `(indices, longueurs, labels)` ;*
- *les trois `DataLoader` (entraînement mélangé, validation et test non mélangés), avec des batchs de 64.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
class TweetDataset(Dataset):
  def __init__(self, split) -> None:
    self.sequences = [torch.tensor(numericalize(text) or [vocab["<unk>"]])
                      for text in split["text"]]
    self.labels = split["label"]

  def __len__(self) -> int:
    return len(self.labels)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
    return self.sequences[i], self.labels[i]


def collate(batch):
  sequences, labels = zip(*batch)
  lengths = torch.tensor([len(sequence) for sequence in sequences])
  padded = pad_sequence(sequences, batch_first=True, padding_value=vocab["<pad>"])
  return padded, lengths, torch.tensor(labels)


train_loader = DataLoader(TweetDataset(tweets["train"]), batch_size=64,
                          shuffle=True, collate_fn=collate)
val_loader = DataLoader(TweetDataset(tweets["validation"]), batch_size=64,
                        collate_fn=collate)
test_loader = DataLoader(TweetDataset(tweets["test"]), batch_size=64,
                         collate_fn=collate)

ids, lengths, labels = next(iter(train_loader))
print(ids.shape, lengths[:8], labels[:8])

## Le modèle : un LSTM bidirectionnel

*Complétez le modèle suivant :*

- *une couche `nn.Embedding` (avec `padding_idx`) ;*
- *un `nn.LSTM` bidirectionnel ;*
- *un dropout, puis une couche linéaire qui produit un **logit** par classe (pas de softmax : `cross_entropy` s'en charge).*

*Dans `forward`, utilisez `pack_padded_sequence` pour que le LSTM ignore le padding, puis concaténez les derniers états cachés des deux directions (`h_n[-2]` et `h_n[-1]`).*

*Attention : `pack_padded_sequence` attend des longueurs sur CPU.*

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    # Votre code ici

  def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    # Votre code ici
    pass

### Solution

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True,
                        bidirectional=True)
    self.dropout = nn.Dropout(0.3)
    self.head = nn.Linear(2 * hidden_size, n_classes)

  def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(ids))
    packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True,
                                  enforce_sorted=False)
    _, (h_n, _) = self.lstm(packed)
    # Derniers états cachés des directions avant et arrière
    summary = torch.cat([h_n[-2], h_n[-1]], dim=1)
    return self.head(self.dropout(summary))


model = SentimentLSTM(len(vocab)).to(device)
print(model)
print(sum(p.numel() for p in model.parameters()), "paramètres")

## Boucle d'évaluation

*Codez une fonction `predict(model, loader) -> tuple[numpy.ndarray, numpy.ndarray]` qui renvoie les labels prédits et les vrais labels de tout un `DataLoader`.*

*N'oubliez pas `model.eval()`, `torch.no_grad()` et de déplacer les tenseurs sur `device`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def predict(model: nn.Module, loader: DataLoader) -> tuple[numpy.ndarray, numpy.ndarray]:
  model.eval()
  predictions, truths = [], []
  with torch.no_grad():
    for ids, lengths, labels in loader:
      logits = model(ids.to(device), lengths)
      predictions.append(logits.argmax(dim=1).cpu())
      truths.append(labels)
  return torch.cat(predictions).numpy(), torch.cat(truths).numpy()


def macro_recall(model: nn.Module, loader: DataLoader) -> float:
  predictions, truths = predict(model, loader)
  return recall_score(truths, predictions, average="macro")


print(f"Rappel moyen avant entraînement : {macro_recall(model, val_loader):.3f}")

## Boucle d'entraînement

*Codez la boucle d'entraînement :*

- *optimiseur `AdamW` avec un learning rate de `1e-3` ;*
- *perte `nn.functional.cross_entropy` ;*
- *à la fin de chaque epoch, calculez le rappel moyen sur la validation ;*
- *gardez une copie des poids du meilleur modèle (`copy.deepcopy(model.state_dict())`) et arrêtez l'entraînement si la validation ne s'améliore plus pendant 2 epochs (*early stopping*) ;*
- *à la fin, rechargez les meilleurs poids avec `load_state_dict`.*

*Une dizaine d'epochs suffit.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
model = SentimentLSTM(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

best_recall, best_state, patience = 0.0, None, 0
for epoch in range(10):
  model.train()
  total_loss = 0.0
  for ids, lengths, labels in train_loader:
    ids, labels = ids.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(ids, lengths), labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  val_recall = macro_recall(model, val_loader)
  print(f"Epoch {epoch + 1} : perte {total_loss / len(train_loader):.3f}, "
        f"rappel moyen validation {val_recall:.3f}")
  if val_recall > best_recall:
    best_recall, best_state, patience = val_recall, copy.deepcopy(model.state_dict()), 0
  else:
    patience += 1
    if patience == 2:
      print("Early stopping")
      break

model.load_state_dict(best_state)
torch.save(model.state_dict(), "sentiment-lstm.pt")
print(f"Meilleur rappel moyen en validation : {best_recall:.3f}")

## Évaluation finale sur l'ensemble de test

*Évaluez le meilleur modèle sur l'ensemble de test : rapport de classification (`classification_report`) et matrice de confusion (`ConfusionMatrixDisplay.from_predictions`). Faites de même pour la référence TF-IDF.*

*Quelles classes sont le plus souvent confondues ? Le LSTM fait-il mieux que la référence ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
predictions, truths = predict(model, test_loader)
print("LSTM bidirectionnel")
print(classification_report(truths, predictions, target_names=label_names, digits=3))
ConfusionMatrixDisplay.from_predictions(truths, predictions, display_labels=label_names,
                                        normalize="true")
plt.show()

baseline_predictions = baseline.predict(vectorizer.transform(tweets["test"]["text"]))
print("Référence TF-IDF")
print(classification_report(truths, baseline_predictions, target_names=label_names, digits=3))

- Les confusions se font surtout avec la classe *neutre* ; *négatif* et *positif* sont rarement confondus entre eux.
- Le LSTM appris depuis zéro ne fait qu'un peu mieux que la référence TF-IDF (rappel moyen d'environ 0,59 contre 0,56 en test) : avec quelques dizaines de milliers de tweets seulement, il manque de données pour apprendre le sens des mots. C'est la motivation des modèles pré-entraînés.

## Pour comparer : un transformeur pré-entraîné

Le modèle [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) est un RoBERTa pré-entraîné sur des millions de tweets, puis affiné sur cette même tâche. La fonction `pipeline` de `transformers` permet de l'utiliser en une ligne.

*Évaluez-le sur les 1 000 premiers tweets de test et comparez son rappel moyen à celui du LSTM sur ces mêmes tweets. Qu'en concluez-vous ?*

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                      device=0 if device == "cuda" else -1)
print(classifier("I love this course on PyTorch!"))

In [ ]:
# Votre code ici

### Solution

In [ ]:
subset = tweets["test"].select(range(1000))
name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
roberta_predictions = [name_to_label[output["label"]]
                       for output in classifier(list(subset["text"]), batch_size=64)]
print(f"RoBERTa pré-entraîné : "
      f"{recall_score(list(subset['label']), roberta_predictions, average='macro'):.3f}")
print(f"LSTM : {recall_score(truths[:1000], predictions[:1000], average='macro'):.3f}")

Le transformeur pré-entraîné fait nettement mieux (environ 0,71 contre 0,59) : il a appris le langage des tweets sur des millions d'exemples avant d'être affiné. Pour une tâche de classification de texte courante, partir d'un modèle pré-entraîné est aujourd'hui le premier réflexe ; entraîner un LSTM depuis zéro reste utile quand les contraintes de calcul sont fortes ou quand aucun modèle pré-entraîné n'existe pour la langue ou le domaine.